In [1]:
from __future__ import annotations

In [2]:
import importlib.util
import sys
import unittest
from pathlib import Path

In [3]:
import numpy as np
import pandas as pd

In [4]:
MODULE_PATH = Path().resolve().parent / "src" / "04_fcm_model.py"
SPEC = importlib.util.spec_from_file_location("fcm_model", MODULE_PATH)
assert SPEC and SPEC.loader
fcm_model = importlib.util.module_from_spec(SPEC)
sys.modules["fcm_model"] = fcm_model
SPEC.loader.exec_module(fcm_model)

In [5]:
class FCMModelTests(unittest.TestCase):
    def test_partition_metrics_are_valid(self) -> None:
        membership = np.array([[0.8, 0.2], [0.4, 0.6], [1.0, 0.0]])
        pc = fcm_model.calculate_partition_coefficient(membership)
        mpc = fcm_model.calculate_modified_partition_coefficient(pc, c=2)
        pe = fcm_model.calculate_partition_entropy(membership)

        self.assertGreaterEqual(pc, 0.5)
        self.assertLessEqual(pc, 1.0)
        self.assertTrue(np.isclose(mpc, (pc - 0.5) / 0.5))
        self.assertTrue(np.isfinite(pe))

    def test_xie_beni_handles_identical_centroids(self) -> None:
        x = np.array([[0.0, 0.0], [1.0, 1.0]])
        centroids = np.array([[0.5, 0.5], [0.5, 0.5]])
        membership = np.array([[0.5, 0.5], [0.5, 0.5]])

        self.assertTrue(np.isinf(fcm_model.calculate_xie_beni(x, centroids, membership, m=2.0)))

    def test_membership_rows_sum_and_crisp_argmax(self) -> None:
        membership = np.array([[0.1, 0.9], [0.7, 0.3]])
        crisp = np.argmax(membership, axis=1) + 1

        self.assertTrue(np.allclose(membership.sum(axis=1), 1.0))
        self.assertEqual(crisp.tolist(), [2, 1])

    def test_label_alignment_preserves_membership_content(self) -> None:
        reference = np.array([[0.0, 0.0], [10.0, 10.0]])
        run = fcm_model.FCMRunResult(
            c=2,
            m=2.0,
            seed=1,
            converged=True,
            iterations=5,
            final_objective=1.0,
            objective_history=[2.0, 1.0],
            centroids=np.array([[10.0, 10.0], [0.0, 0.0]]),
            membership=np.array([[0.2, 0.8], [0.9, 0.1]]),
            crisp_cluster=np.array([1, 0]),
            maximum_membership=np.array([0.8, 0.9]),
            xie_beni=0.1,
            partition_coefficient=0.75,
            modified_partition_coefficient=0.5,
            partition_entropy=0.3,
            minimum_centroid_distance=200.0,
        )

        aligned = fcm_model.align_cluster_labels(reference, run)

        self.assertTrue(np.allclose(aligned.centroids, reference))
        self.assertTrue(np.allclose(np.sort(aligned.membership, axis=1), np.sort(run.membership, axis=1)))
        self.assertTrue(np.allclose(aligned.membership.sum(axis=1), 1.0))

    def test_incomplete_input_raises_informative_error(self) -> None:
        bad_path = Path(__file__).parent / "bad_input_for_fcm_test.csv"
        try:
            pd.DataFrame({"province_name": ["A"], "maternal_age_risk_z": [0.0]}).to_csv(
                bad_path, index=False
            )
            with self.assertRaisesRegex(ValueError, "missing required columns"):
                fcm_model.load_and_validate_data(bad_path)
        finally:
            if bad_path.exists():
                bad_path.unlink()

In [6]:
if __name__ == "__main__":
    unittest.main()

usage: ipykernel_launcher.py [-h] [-v] [-q] [--locals] [--durations N] [-f]
                             [-c] [-b] [-k TESTNAMEPATTERNS]
                             [tests ...]
ipykernel_launcher.py: error: argument -f/--failfast: ignored explicit argument 'c:\\Users\\hadis\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3099d9be9ad4c6e3ab9f0d572920db225a452317f.json'


SystemExit: 2

C:\Users\hadis\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
